# 🔍 **Key Differences Between SQLite and MySQL**

---

### 🔧 **1. Architecture**

| Feature | **SQLite**                                         | **MySQL**                               |
| ------- | -------------------------------------------------- | --------------------------------------- |
| Type    | Embedded (serverless) database engine              | Client-server relational database       |
| Setup   | No installation required; uses a single `.db` file | Requires installation and configuration |
| Storage | Local file-based                                   | Server-based (can be remote or local)   |

---

### ⚙️ **2. Performance & Scalability**

| Aspect      | **SQLite**                                        | **MySQL**                                              |
| ----------- | ------------------------------------------------- | ------------------------------------------------------ |
| Concurrency | Limited (single writer at a time)                 | High (supports many concurrent users and threads)      |
| Speed       | Faster for lightweight/local applications         | Optimized for multi-user and large-scale systems       |
| Scalability | Not suitable for high-load or distributed systems | Scales well with replication, clustering, and sharding |

---

### 🔐 **3. Features**

| Feature           | **SQLite**                     | **MySQL**                                          |
| ----------------- | ------------------------------ | -------------------------------------------------- |
| Transactions      | ACID compliant | ACID compliant                                     |
| Authentication    | No built-in user management    | Full user/role-based authentication and privileges |
| Stored Procedures | Not supported                  | Supported                                          |
| Triggers          | Supported                      | Supported                                          |

---

### 📦 **4. Use Cases**

| Context  | **SQLite**                               | **MySQL**                                               |
| -------- | ---------------------------------------- | ------------------------------------------------------- |
| Best For | Mobile apps, IoT, desktop tools, testing | Web apps, enterprise apps, high-load production systems |
| Examples | Firefox, Android, small tools            | WordPress, Shopify, large web services                  |

---

### 🔄 **5. Portability & Maintenance**

| Feature     | **SQLite**                             | **MySQL**                                  |
| ----------- | -------------------------------------- | ------------------------------------------ |
| Portability | Very portable—just copy the `.db` file | Requires export/import or dump/restore     |
| Backup      | Simple file copy                       | Requires structured backup strategies      |
| Maintenance | Minimal                                | Requires periodic optimization and updates |

---

### 📚 **Official Resources**

- SQLite: [https://www.sqlite.org/](https://www.sqlite.org/)
- MySQL: [https://www.mysql.com/](https://www.mysql.com/)

---

### ✅ **Summary**

| Criteria    | SQLite                        | MySQL                          |
| ----------- | ----------------------------- | ------------------------------ |
| Lightweight | ✅                             | ❌ (heavier setup)              |
| Concurrency | ❌ (limited)                   | ✅ (multi-user support)         |
| Use Case    | Local, testing, embedded apps | Large-scale, web-based systems |

> Choose **SQLite** for simplicity and portability, and **MySQL** for complex, scalable, and secure multi-user applications.




---



---



---



# **Install Dependencies and Create Directories**

- **`fpdf`**  
  A library for generating PDF documents.

- **`faker`**  
  A library for generating fake data, which is used here to populate a sample database.

- **`streamlit`**  
  A library for building interactive web applications, which will be used later to create a dashboard.

- **`pandas`**  
  A powerful library for data manipulation and analysis, commonly used for working with tabular data.

- **`matplotlib`**  
  A fundamental plotting library for creating static, interactive, and animated visualizations in Python.

- **`seaborn`**  
  A statistical data visualization library based on `matplotlib`, providing a high-level interface for drawing attractive and informative statistical graphics.


In [1]:
!pip install fpdf faker streamlit pandas matplotlib seaborn


In [2]:
import os

# Create the 'app' directory if it doesn't exist
if not os.path.exists('app'):
    os.makedirs('app')

# Create the 'gui' directory if it doesn't exist
if not os.path.exists('gui'):
    os.makedirs('gui')

# Create the 'static' directory if it doesn't exist
if not os.path.exists('static'):
    os.makedirs('static')
    os.makedirs('static/charts')
    os.makedirs('static/reports')

print("Dependencies installed and directories created.")

Dependencies installed and directories created.


# **Create Database and Application Files**

In [3]:
import sqlite3 #Python library for working with SQLite databases
from faker import Faker
import datetime
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from fpdf import FPDF
import base64
from IPython.display import HTML, display

# Initialize Faker
fake = Faker()

# Database connection
DATABASE_NAME = 'hospital.db'

def create_connection():
    conn = None
    try:
        conn = sqlite3.connect(DATABASE_NAME)
    except sqlite3.Error as e:
        print(e)
    return conn

# This function is designed to interact with a SQLite database
def execute_query(conn, query, params=None):
    try:
        cur = conn.cursor()
        if params:
            cur.execute(query, params)
        else:
            cur.execute(query)
        conn.commit() # Commit after each successful query
        return cur
    except sqlite3.Error as e:
        print(f"Database error: {e}")
        return None

# retrieve information from the SQLite database
def fetch_data(conn, query, params=None):
    try:
        cur = conn.cursor()
        if params:
            cur.execute(query, params)
        else:
            cur.execute(query)
        rows = cur.fetchall()
        return rows
    except sqlite3.Error as e:
        print(f"Database error: {e}")
        return None

# Functions to create and populate data (simplified from previous version)
def create_patient(conn):
    sql = '''INSERT INTO Patient(name, dob, gender, contact_info) VALUES(?,?,?,?)'''
    patient = (fake.name(), fake.date_of_birth(minimum_age=18, maximum_age=80).strftime('%Y-%m-%d'), fake.random_element(elements=('Male', 'Female')), fake.phone_number())
    cur = conn.cursor()
    cur.execute(sql, patient)
    # conn.commit() # No need to commit here, commit happens in execute_query or after batch inserts
    return cur.lastrowid

def create_doctor(conn):
    sql = '''INSERT INTO Doctor(name, specialization, email, phone) VALUES(?,?,?,?)'''
    doctor = (fake.name(), fake.job(), fake.email(), fake.phone_number())
    cur = conn.cursor()
    cur.execute(sql, doctor)
    # conn.commit() # No need to commit here
    return cur.lastrowid

def create_room(conn):
    sql = '''INSERT INTO Room(room_number, type, status) VALUES(?,?,?)'''
    room = (fake.building_number(), fake.random_element(elements=('ICU', 'General', 'Operating')), fake.random_element(elements=('Available', 'Occupied')))
    cur = conn.cursor()
    cur.execute(sql, room)
    # conn.commit() # No need to commit here
    return cur.lastrowid

def create_appointment(conn, patient_id, doctor_id):
    sql = '''INSERT INTO Appointment(patient_id, doctor_id, appointment_date, status) VALUES(?,?,?,?)'''
    appointment = (patient_id, doctor_id, fake.future_date(end_date='+30d').strftime('%Y-%m-%d'), fake.random_element(elements=('Scheduled', 'Completed', 'Cancelled')))
    cur = conn.cursor()
    cur.execute(sql, appointment)
    # conn.commit() # No need to commit here
    return cur.lastrowid

def create_admission(conn, patient_id, room_id):
    sql = '''INSERT INTO Admission(patient_id, room_id, admission_date, discharge_date) VALUES(?,?,?,?)'''
    admission_date = fake.date_between(start_date='-30d', end_date='today').strftime('%Y-%m-%d')
    discharge_date = (datetime.datetime.strptime(admission_date, '%Y-%m-%d') + datetime.timedelta(days=fake.random_int(min=1, max=14))).strftime('%Y-%m-%d')
    admission = (patient_id, room_id, admission_date, discharge_date)
    cur = conn.cursor()
    cur.execute(sql, admission)
    # conn.commit() # No need to commit here
    return cur.lastrowid


# create_db() is responsible for setting up the SQLite database named 'hospital.db'
def create_db():
    # Remove existing database file to ensure clean creation
    if os.path.exists(DATABASE_NAME):
        os.remove(DATABASE_NAME)
        print(f"Removed existing database: {DATABASE_NAME}")

    conn = create_connection()
    if conn is not None:
        # Create tables
        sql_script = '''
        CREATE TABLE Patient (
            patient_id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            dob TEXT,
            gender TEXT,
            contact_info TEXT
        );

        CREATE TABLE Doctor (
            doctor_id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            specialization TEXT,
            email TEXT,
            phone TEXT
        );

        CREATE TABLE Room (
            room_id INTEGER PRIMARY KEY AUTOINCREMENT,
            room_number TEXT NOT NULL,
            type TEXT,
            status TEXT
        );

        CREATE TABLE Appointment (
            appointment_id INTEGER PRIMARY KEY AUTOINCREMENT,
            patient_id INTEGER,
            doctor_id INTEGER,
            appointment_date TEXT,
            status TEXT,
            FOREIGN KEY (patient_id) REFERENCES Patient(patient_id),
            FOREIGN KEY (doctor_id) REFERENCES Doctor(doctor_id)
        );


        CREATE TABLE IF NOT EXISTS AuditLog (
            log_id INTEGER PRIMARY KEY AUTOINCREMENT,
            user TEXT,
            action TEXT,
            table_name TEXT,
            timestamp TEXT
        );

        CREATE TABLE Admission (
            admission_id INTEGER PRIMARY KEY AUTOINCREMENT,
            patient_id INTEGER,
            room_id INTEGER,
            admission_date TEXT,
            discharge_date TEXT,
            FOREIGN KEY (patient_id) REFERENCES Patient(patient_id),
            FOREIGN KEY (room_id) REFERENCES Room(room_id)
        );
        '''
        try:
            # Execute table creation script
            conn.executescript(sql_script) # Use executescript for multiple statements
            conn.commit() # Commit the table creation
            print("Tables created.")
        except sqlite3.OperationalError as e:
            print(f"Error creating tables: {e}") # Print error for debugging
            print("Tables might already exist or there was an issue.") # Provide feedback
            conn.rollback() # Rollback if table creation fails
            conn.close()
            return # Exit if table creation failed

        # Populate tables if empty
        # This check is now reliable because the DB is recreated if it exists
        if not fetch_data(conn, "SELECT 1 FROM Patient LIMIT 1"):
            print("Seeding data...")
            num_patients = 50
            num_doctors = 10
            num_rooms = 20
            num_appointments = 100
            num_admissions = 50

            # Use a single cursor for batch insertions and commit at the end
            cur = conn.cursor()

            patients = [create_patient(conn) for _ in range(num_patients)]
            doctors = [create_doctor(conn) for _ in range(num_doctors)]
            rooms = [create_room(conn) for _ in range(num_rooms)]

            for _ in range(num_appointments):
                patient_id = fake.random_element(elements=patients)
                doctor_id = fake.random_element(elements=doctors)
                create_appointment(conn, patient_id, doctor_id)

            for _ in range(num_admissions):
                patient_id = fake.random_element(elements=patients)
                room_id = fake.random_element(elements=rooms)
                create_admission(conn, patient_id, room_id)

            conn.commit() # Commit all insertions at once
            print("Data seeding complete.")
        else:
             # This block is less likely to be hit with the os.remove call,
             # but good to keep for robustness.
            print("Database already contains data. Skipping seeding.")


    else:
        print("Cannot create the database connection.")

    if conn:
        conn.close()


def create_app_files():
    # Create app directory
    if not os.path.exists('app'):
        os.makedirs('app')



### 🧾 **Overview of the Code**

This Python script sets up a **simulated hospital management database** (`hospital.db`) using **SQLite** and populates it with **realistic fake data** using the [`Faker`](https://faker.readthedocs.io/) library.  
It also prepares an `app/` directory to support **modular application development**.

---

### 🧱 **Key Functional Components**

#### ✅ 1. **Database Initialization**

- **`DATABASE_NAME`**: `'hospital.db'` — the SQLite database file
- **`create_connection()`**: Establishes connection to the database
- **`create_db()`**:
  - Deletes any existing `hospital.db` file to start fresh
  - Creates five main tables:
    - `Patient`
    - `Doctor`
    - `Room`
    - `Appointment`
    - `Admission`
  - Seeds data only if database is empty

---

### 🧪 **Fake Data Generation Functions**

Each function inserts one record using **Faker-generated data**:

| Function               | Inserts Into  | Notes                                    |
| ---------------------- | ------------- | ---------------------------------------- |
| `create_patient()`     | `Patient`     | Name, DOB, gender, phone                 |
| `create_doctor()`      | `Doctor`      | Name, specialization, email, phone       |
| `create_room()`        | `Room`        | Room number, type, status                |
| `create_appointment()` | `Appointment` | Links a patient and doctor with date     |
| `create_admission()`   | `Admission`   | Links a patient and room with date range |

- All queries use **parameterized SQL** to prevent injection

---

### 📋 **Database Schema Summary**

| Table         | Key Fields                                         | Foreign Keys              |
| ------------- | -------------------------------------------------- | ------------------------- |
| `Patient`     | `patient_id`, `name`, `dob`                        | —                         |
| `Doctor`      | `doctor_id`, `specialization`                      | —                         |
| `Room`        | `room_id`, `type`, `status`                        | —                         |
| `Appointment` | `appointment_id`, `date`, `status`                 | `patient_id`, `doctor_id` |
| `Admission`   | `admission_id`, `admission_date`, `discharge_date` | `patient_id`, `room_id`   |

---

### 🗃️ **Batch Processing and Optimization**

- Uses `conn.executescript()` to define the schema efficiently  
- Performs **bulk inserts** using a shared cursor and a single `commit()`  
- Includes robust **error handling** and informative logging

---

### 📂 **Directory Management**

- **`create_app_files()`**: Ensures the `app/` directory is created to store modular files like:
  - `db_utils.py`
  - `analytics.py`
  - `report_generator.py`

---

### ✅ **Summary**

| Category            | Details                                                                 |
| ------------------- | ----------------------------------------------------------------------- |
| Purpose             | Create and seed a SQLite hospital database                              |
| Libraries Used      | `sqlite3`, `Faker`, `os`, `datetime`, `pandas`, `matplotlib`, `seaborn` |
| Tables Created      | 5 (`Patient`, `Doctor`, `Room`, `Appointment`, `Admission`)             |
| Fake Data Generated | 50 patients, 10 doctors, 20 rooms, 100 appointments, 50 admissions      |
| Safety              | Parameterized SQL, exception handling, clean-up before creation         |

> 💡 **Use Case**: This script is ideal for building educational tools, dashboards, or prototypes involving healthcare data in a local environment.


In [4]:
%%writefile app/db_utils.py
import sqlite3

DATABASE_NAME = 'hospital.db'

def create_connection():
    conn = None
    try:
        conn = sqlite3.connect(DATABASE_NAME)
    except sqlite3.Error as e:
        print(e)
    return conn

def execute_query(conn, query, params=None):
    try:
        cur = conn.cursor()
        if params:
            cur.execute(query, params)
        else:
            cur.execute(query)
        conn.commit()
        return cur
    except sqlite3.Error as e:
        print(f"Database error: {e}")
        return None

def fetch_data(conn, query, params=None):
    try:
        cur = conn.cursor()
        if params:
            cur.execute(query, params)
        else:
            cur.execute(query)
        rows = cur.fetchall()
        return rows
    except sqlite3.Error as e:
        print(f"Database error: {e}")
        return None


Writing app/db_utils.py


### 🧾 **Overview of `app/db_utils.py`**

This module defines the core **database utility functions** for interacting with a **SQLite** database (`hospital.db`).  
It encapsulates connection handling, query execution, and data retrieval with proper error management and secure query practices.

---

### 🔧 **Functions Defined**

---

#### 1. `create_connection()`

- **Purpose**: Establishes and returns a connection to the SQLite database.
- **Error Handling**: Catches and prints `sqlite3.Error` if connection fails.
- **Returns**: A `sqlite3.Connection` object or `None`.

---

#### 2. `execute_query(conn, query, params=None)`

- **Purpose**: Executes a given SQL command (e.g., INSERT, UPDATE, DELETE).
- **Parameters**:
  - `conn`: SQLite connection
  - `query`: SQL command string
  - `params`: Tuple of parameters for secure, parameterized queries
- **Security**: Uses `?` placeholders to prevent SQL injection
- **Returns**: Cursor object if successful, otherwise `None`

---

#### 3. `fetch_data(conn, query, params=None)`

- **Purpose**: Executes a SELECT query and returns the results.
- **Parameters**:
  - `conn`: SQLite connection
  - `query`: SQL SELECT statement
  - `params`: Optional filter parameters
- **Returns**: A list of rows (`List[Tuple]`) or `None` on failure

---

### ✅ **Summary Table**

| Function            | Description                         | Returns                 | SQL Injection Safe |
| ------------------- | ----------------------------------- | ----------------------- | ------------------ |
| `create_connection` | Connects to SQLite DB               | `Connection`            | N/A                |
| `execute_query`     | Executes INSERT/UPDATE/DELETE       | `Cursor` or `None`      | ✅                  |
| `fetch_data`        | Executes SELECT and returns results | `List[Tuple]` or `None` | ✅                  |

---

### 📌 **Use Case**

These utility functions serve as the **backend database layer** for interacting with `hospital.db` in:

- 📊 `gui/dashboard.py` – Streamlit dashboard  
- 📈 `analytics.py` – Chart generation  
- 📝 `report_generator.py` – PDF report creation

They enable **clean**, **modular**, and **secure** database access for all components of the application.


In [5]:
%%writefile app/query_functions.py
import sqlite3
from datetime import datetime, timedelta
from .db_utils import fetch_data  # Corrected import

# Retrieves upcoming appointments for a given doctor within a specified number of days (default: 7)
def fetch_upcoming_appointments_per_doctor(conn, doctor_id, days=7):
    start_date = datetime.now().strftime('%Y-%m-%d')
    end_date = (datetime.now() + timedelta(days=days)).strftime('%Y-%m-%d')
    sql = """
        SELECT * FROM Appointment
        WHERE doctor_id = ?
        AND appointment_date BETWEEN ? AND ?
        ORDER BY appointment_date
    """
    params = (doctor_id, start_date, end_date)
    return fetch_data(conn, sql, params)

# Calculates the percentage of occupied rooms, optionally filtered by room type
def calculate_room_occupancy_rate(conn, room_type=None):
    sql = """
        SELECT
            COUNT(CASE WHEN status = 'Occupied' THEN 1 END) AS occupied_rooms,
            COUNT(*) AS total_rooms
        FROM Room
    """
    params = None
    if room_type:
        sql += " WHERE type = ?"
        params = (room_type,)
    rooms_data = fetch_data(conn, sql, params)
    if rooms_data:
        occupied_rooms, total_rooms = rooms_data[0]
        if total_rooms is not None and total_rooms > 0:
            occupancy_rate = (occupied_rooms / total_rooms) * 100
            return occupancy_rate
        else:
            return 0
    else:
        return None

# Computes the average stay duration (in days) of patients per room type using JULIANDAY() function
def compute_average_patient_stay_by_room_type(conn, room_type):
    sql = """
        SELECT AVG(JULIANDAY(discharge_date) - JULIANDAY(admission_date))
        FROM Admission
        JOIN Room ON Admission.room_id = Room.room_id
        WHERE Room.type = ?
    """
    params = (room_type,)
    stay_data = fetch_data(conn, sql, params)
    if stay_data and stay_data[0] and stay_data[0][0] is not None:
        average_stay = stay_data[0][0]
        return average_stay
    else:
        return None
# Identifies doctors who have more than a threshold number of appointments on a given day
def detect_overbooked_doctors(conn, appointment_threshold=5):
    sql = """
        SELECT doctor_id, appointment_date, COUNT(*) AS appointment_count
        FROM Appointment
        GROUP BY doctor_id, appointment_date
        HAVING COUNT(*) > ?
    """
    params = (appointment_threshold,)
    return fetch_data(conn, sql, params)

Writing app/query_functions.py


### 🔍 **Overview of the Code Snippet**

This module (`query_functions.py`) defines **analytical SQL query functions** for a hospital management system using a **SQLite** database. It imports a helper function `fetch_data` from `db_utils.py` to execute parameterized SQL queries. Below is a brief description of each function:

---

#### 1. **`fetch_upcoming_appointments_per_doctor`**

- Retrieves upcoming appointments for a specific doctor within a given number of days (default: 7).
- Filters results between `today` and `today + days`.

---

#### 2. **`calculate_room_occupancy_rate`**

- Calculates the percentage of occupied rooms, optionally filtered by `room_type`.
- Adds a conditional filter if a specific room type is provided.

---

#### 3. **`compute_average_patient_stay_by_room_type`**

- Computes the **average length of stay (in days)** per room type.
- Uses the SQLite `JULIANDAY()` function for date difference calculation.

---

#### 4. **`detect_overbooked_doctors`**

- Identifies doctors who have more than a specified threshold of appointments on a particular date.

---

###  **🛡️IMPORTANT: SQL Injection Prevention Mechanism**

This code follows **best practices** to prevent SQL injection through **parameterized queries**, facilitated by the `fetch_data()` function.

**Key Safeguards:**

- ✅ **Safe Parameter Binding**  
  Uses `?` placeholders to bind user inputs securely:
  ```python
  sql = "SELECT * FROM Appointment WHERE doctor_id = ?"
  params = (doctor_id,)


### 🛡️ SQL Injection Safety Highlights

---

#### ✅ **No Direct String Interpolation**

The code **avoids dangerous patterns** that can lead to SQL injection:

- ❌ `f-strings` with SQL code  
- ❌ `%`-formatting of SQL strings  
- ❌ String concatenation using `+`

---

#### ✅ **Delegated Execution**

User input is **never embedded directly** into SQL statements. Instead:

- Parameters are passed **separately** to `cursor.execute()`  
- The SQLite engine handles **automatic escaping** and secure substitution

```python
sql = "SELECT * FROM Appointment WHERE doctor_id = ?"
params = (doctor_id,)
cursor.execute(sql, params)




---



---



### 📅 **`JULIANDAY()` Function in SQLite**

The **`JULIANDAY()`** function in **SQLite** returns the **Julian day number**, which is a **floating-point value** representing the number of days (Julian calendar).

---

### 🔹 **Usage Purpose**

- Commonly used to **calculate date differences**, such as durations between two dates in days.

---

### 🔹 **Syntax**

```sql
JULIANDAY('YYYY-MM-DD HH:MM:SS')




### **Example**
```sql
SELECT JULIANDAY('2025-05-27') - JULIANDAY('2025-05-20');

-- Output: 7.0


This returns the difference in days between May 27, 2025 and May 20, 2025.





---



---



In [6]:
%%writefile app/analytics.py
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from .db_utils import fetch_data  # Corrected import

def get_appointments_by_specialization(conn):
    sql = """
        SELECT d.specialization, COUNT(a.appointment_id) AS num_appointments
        FROM Appointment a
        JOIN Doctor d ON a.doctor_id = d.doctor_id
        GROUP BY d.specialization
    """
    data = fetch_data(conn, sql)
    if data:
        df = pd.DataFrame(data, columns=['specialization', 'num_appointments'])
        plt.figure(figsize=(10, 6))
        sns.barplot(x='specialization', y='num_appointments', data=df)
        plt.title('Number of Appointments by Specialization')
        plt.xlabel('Specialization')
        plt.ylabel('Number of Appointments')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        if not os.path.exists('static/charts'):
            os.makedirs('static/charts')
        plt.savefig('static/charts/appointments_by_specialization.png')
        plt.close()
        return 'static/charts/appointments_by_specialization.png'
    else:
        return None

def get_patient_gender_distribution(conn):
    sql = """
        SELECT gender, COUNT(*) AS count FROM Patient GROUP BY gender
    """
    data = fetch_data(conn, sql)
    if data:
        df = pd.DataFrame(data, columns=['gender', 'count'])
        plt.figure(figsize=(8, 6))
        plt.pie(df['count'], labels=df['gender'], autopct='%1.1f%%', startangle=140)
        plt.title('Patient Gender Distribution')
        if not os.path.exists('static/charts'):
            os.makedirs('static/charts')
        plt.savefig('static/charts/patient_gender_distribution.png')
        plt.close()
        return 'static/charts/patient_gender_distribution.png'
    else:
        return None

def get_room_usage_trends(conn):
    sql = """
        SELECT admission_date, COUNT(DISTINCT room_id) AS rooms_used
        FROM Admission
        GROUP BY admission_date
        ORDER BY admission_date
    """
    data = fetch_data(conn, sql)
    if data:
        df = pd.DataFrame(data, columns=['admission_date', 'rooms_used'])
        df['admission_date'] = pd.to_datetime(df['admission_date'])
        plt.figure(figsize=(12, 6))
        plt.plot(df['admission_date'], df['rooms_used'], marker='o')
        plt.title('Time-Series Room Usage Trends')
        plt.xlabel('Admission Date')
        plt.ylabel('Number of Rooms Used')
        plt.grid(True)
        plt.tight_layout()
        if not os.path.exists('static/charts'):
            os.makedirs('static/charts')
        plt.savefig('static/charts/room_usage_trends.png')
        plt.close()
        return 'static/charts/room_usage_trends.png'
    else:
        return None


Writing app/analytics.py


### 📊 **Overview of `app/analytics.py`**

This Python module provides **visual analytics for hospital data** by generating charts from a **SQLite** database. It uses `pandas`, `matplotlib`, and `seaborn` for data handling and visualization.  
The `fetch_data` function (imported from `db_utils`) is used to query the database.

---

### 🔧 **Functions and Their Purposes**

---

#### 1. `get_appointments_by_specialization(conn)`

- **Purpose**: Visualizes how many appointments are scheduled per medical specialization.
- **Process**:
  - Joins `Appointment` and `Doctor` tables.
  - Groups results by `specialization`.
  - Generates a **bar chart** saved as:  
    `static/charts/appointments_by_specialization.png`

---

#### 2. `get_patient_gender_distribution(conn)`

- **Purpose**: Displays gender distribution among registered patients.
- **Process**:
  - Groups patients by `gender` and counts each group.
  - Generates a **pie chart** saved as:  
    `static/charts/patient_gender_distribution.png`

---

#### 3. `get_room_usage_trends(conn)`

- **Purpose**: Tracks hospital room usage trends over time.
- **Process**:
  - Groups `Admission` records by `admission_date`.
  - Counts distinct `room_id` per date.
  - Generates a **line chart** saved as:  
    `static/charts/room_usage_trends.png`

---

### 🗂️ **Storage of Charts**

- All charts are saved to the directory:  
  `static/charts/`
- If the directory does not exist, it is created using `os.makedirs()`.

---

### 📌 **Summary**

| Feature        | Description                              |
| -------------- | ---------------------------------------- |
| Input          | SQLite connection object                 |
| Output         | PNG images of charts                     |
| Libraries Used | `pandas`, `matplotlib`, `seaborn`, `os`  |
| Application    | Data visualization for hospital insights |

These functions are designed to support **interactive dashboards** (e.g., using **Streamlit**) or to generate **automated reports** (e.g., via PDF).


In [7]:
%%writefile app/report_generator.py
from fpdf import FPDF
import base64
from IPython.display import HTML, display
import os
from .db_utils import create_connection
from .analytics import get_appointments_by_specialization, get_patient_gender_distribution, get_room_usage_trends

def generate_hospital_report():
    conn = create_connection()
    if conn:
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font("Arial", size=16)
        pdf.cell(200, 10, txt="Hospital Analytics Report", ln=1, align="C")

        appointments_chart = get_appointments_by_specialization(conn)
        if appointments_chart and os.path.exists(appointments_chart):
            pdf.image(appointments_chart, x=10, y=30, w=100)
        else:
            pdf.cell(200, 10, txt="Appointments by Specialization chart not available.", ln=1)

        gender_chart = get_patient_gender_distribution(conn)
        if gender_chart and os.path.exists(gender_chart):
            pdf.image(gender_chart, x=120, y=30, w=80)
        else:
            pdf.cell(200, 10, txt="Patient Gender Distribution chart not available.", ln=1)

        room_usage_chart = get_room_usage_trends(conn)
        if room_usage_chart and os.path.exists(room_usage_chart):
            pdf.image(room_usage_chart, x=10, y=100, w=150)
        else:
            pdf.cell(200, 10, txt="Room Usage Trends chart not available.", ln=1)

        if not os.path.exists('static/reports'):
            os.makedirs('static/reports')

        pdf.output("static/reports/hospital_report.pdf")
        conn.close()
        return "static/reports/hospital_report.pdf"
    else:
        return None

def display_pdf(pdf_path):
    with open(pdf_path, "rb") as f:
        base64_pdf = base64.b64encode(f.read()).decode('utf-8')
    pdf_display = HTML('<iframe src="data:application/pdf;base64,{}" width=800 height=600></iframe>'.format(base64_pdf))
    display(pdf_display)


Writing app/report_generator.py



### 📝 **Overview of `app/report_generator.py`**

This module automates the **generation and display of a hospital analytics PDF report**.  
It uses the `FPDF` library for PDF creation and leverages `base64` encoding with `IPython.display` to render the PDF inline in Jupyter/Colab environments.

---

### 📦 **Key Components**

#### 🔹 **Imports**

- `FPDF`: For creating PDF documents  
- `base64`, `HTML`, `display`: For inline rendering of the PDF  
- `os`: For file system operations  
- `create_connection`: From `db_utils`, used to connect to the SQLite database  
- `analytics` functions: Generate charts that are embedded in the report

---

### 🔧 **Function Summaries**

#### 1. `generate_hospital_report()`

- **Purpose**: Generates a PDF report containing hospital analytics charts.
- **Steps**:
  1. Connects to the database
  2. Adds a title to the PDF
  3. Includes up to three visual charts:
     - Appointments by Specialization  
     - Patient Gender Distribution  
     - Room Usage Trends
  4. Embeds each chart only if its corresponding `.png` file exists
  5. Saves the final report to:  
     `static/reports/hospital_report.pdf`
- **Returns**: File path to the generated PDF

---

#### 2. `display_pdf(pdf_path)`

- **Purpose**: Displays the PDF inline in a Jupyter/Colab notebook
- **Steps**:
  1. Reads the PDF file as binary
  2. Base64-encodes the file content
  3. Embeds the file in an `<iframe>` for inline viewing

---

### 📁 **Output**

- PDF report is saved in:  
  `static/reports/`
- Example output file:  
  `hospital_report.pdf`

---

### ✅ **Summary**

| Feature                 | Description                                         |
| ----------------------- | --------------------------------------------------- |
| Input                   | SQLite database connection                          |
| Output                  | PDF report with embedded charts                     |
| Visualizations Included | 3 chart types from `analytics.py`                   |
| Environment Integration | Inline rendering in Jupyter via `display_pdf()`     |
| Libraries Used          | `FPDF`, `base64`, `IPython.display`, `os`           |

> This module is ideal for **automated reporting** in dashboards or interactive notebooks, providing a professional and downloadable summary of hospital data.
```


In [8]:
%%writefile gui/manage_records.py
import streamlit as st
from app.db_utils import create_connection, execute_query, fetch_data

def app():
    st.title("📝 Manage Hospital Records")

    conn = create_connection()

    table_choice = st.selectbox("Choose table to manage", ["Patient", "Doctor", "Room"])

    # Helper: Fetch all records
    def load_records(table):
        return fetch_data(conn, f"SELECT * FROM {table}")

    # Helper: Build form dynamically
    def build_form(table, is_update=False):
        if table == "Patient":
            name = st.text_input("Name")
            dob = st.date_input("Date of Birth")
            gender = st.selectbox("Gender", ["Male", "Female"])
            contact = st.text_input("Contact Info")
            if is_update:
                return {"name": name, "dob": dob, "gender": gender, "contact_info": contact}
            return (name, dob.strftime('%Y-%m-%d'), gender, contact)

        elif table == "Doctor":
            name = st.text_input("Name")
            specialization = st.text_input("Specialization")
            email = st.text_input("Email")
            phone = st.text_input("Phone")
            if is_update:
                return {"name": name, "specialization": specialization, "email": email, "phone": phone}
            return (name, specialization, email, phone)

        elif table == "Room":
            number = st.text_input("Room Number")
            type_ = st.selectbox("Room Type", ["ICU", "General", "Operating"])
            status = st.selectbox("Status", ["Available", "Occupied"])
            if is_update:
                return {"room_number": number, "type": type_, "status": status}
            return (number, type_, status)

    st.subheader(f"{table_choice} Records")
    records = load_records(table_choice)
    st.dataframe(records)

    operation = st.radio("Select operation", ["Add", "Update", "Delete"])

    if operation == "Add":
        st.markdown(f"### ➕ Add new {table_choice}")
        inputs = build_form(table_choice)
        if st.button("Submit"):
            if table_choice == "Patient":
                query = "INSERT INTO Patient(name, dob, gender, contact_info) VALUES (?, ?, ?, ?)"
            elif table_choice == "Doctor":
                query = "INSERT INTO Doctor(name, specialization, email, phone) VALUES (?, ?, ?, ?)"
            elif table_choice == "Room":
                query = "INSERT INTO Room(room_number, type, status) VALUES (?, ?, ?)"
            execute_query(conn, query, inputs)
            st.success(f"{table_choice} added successfully.")

    elif operation == "Update":
        st.markdown(f"### ✏️ Update existing {table_choice}")
        record_id = st.number_input(f"Enter {table_choice} ID to update", min_value=1)
        new_values = build_form(table_choice, is_update=True)
        if st.button("Update"):
            if table_choice == "Patient":
                query = "UPDATE Patient SET name=?, dob=?, gender=?, contact_info=? WHERE patient_id=?"
                params = (*new_values.values(), record_id)
            elif table_choice == "Doctor":
                query = "UPDATE Doctor SET name=?, specialization=?, email=?, phone=? WHERE doctor_id=?"
                params = (*new_values.values(), record_id)
            elif table_choice == "Room":
                query = "UPDATE Room SET room_number=?, type=?, status=? WHERE room_id=?"
                params = (*new_values.values(), record_id)
            execute_query(conn, query, params)
            st.success(f"{table_choice} updated successfully.")

    elif operation == "Delete":
        st.markdown(f"### 🗑️ Delete {table_choice}")
        record_id = st.number_input(f"Enter {table_choice} ID to delete", min_value=1)
        if st.button("Delete"):
            if table_choice == "Patient":
                query = "DELETE FROM Patient WHERE patient_id=?"
            elif table_choice == "Doctor":
                query = "DELETE FROM Doctor WHERE doctor_id=?"
            elif table_choice == "Room":
                query = "DELETE FROM Room WHERE room_id=?"
            execute_query(conn, query, (record_id,))
            st.success(f"{table_choice} deleted successfully.")


Writing gui/manage_records.py


In [9]:
%%writefile gui/dashboard.py
import streamlit as st
import pandas as pd
import sqlite3
from datetime import datetime, timedelta
import sys
import os

# Add the parent directory of 'app' to the Python path
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

import app.db_utils as db_utils
import app.query_functions as query_functions
import app.analytics as analytics
import app.report_generator as report_generator
#from gui import manage_records


# Streamlit GUI
def main():
    st.title("Hospital Analytics & Management Dashboard")
    st.sidebar.title("Navigation")

    # Manage Records Option is added to the GUI
    page = st.sidebar.radio("Go to", ["Dashboard", "Appointments", "Admissions", "Analytics", "Export Report"])

    if page == "Dashboard":
        st.header("Dashboard")
        st.write("Welcome to the Hospital Management System!")
        conn = db_utils.create_connection()
        if conn:
            num_patients = len(db_utils.fetch_data(conn, "SELECT * FROM Patient"))
            st.metric("Number of Patients", num_patients)
            num_doctors = len(db_utils.fetch_data(conn, "SELECT * FROM Doctor"))
            st.metric("Number of Doctors", num_doctors)
            num_rooms = len(db_utils.fetch_data(conn, "SELECT * FROM Room"))
            st.metric("Number of Rooms", num_rooms)
            conn.close()
        else:
            st.error("Failed to connect to the database.")

    elif page == "Appointments":
        st.header("Appointments")
        conn = db_utils.create_connection()
        if conn:
            doctor_ids = db_utils.fetch_data(conn, "SELECT doctor_id, name FROM Doctor")
            doctor_options = {name: id for id, name in doctor_ids}
            selected_doctor_name = st.selectbox("Select Doctor", ["Select a Doctor"] + list(doctor_options.keys()))
            appointments = []

            if selected_doctor_name != "Select a Doctor":
                doctor_id = doctor_options[selected_doctor_name]
                appointment_date = st.date_input("Select Date", datetime.now().date())
                appointments = db_utils.fetch_data(conn, """
                    SELECT a.appointment_id, p.name, d.name, a.appointment_date, a.status
                    FROM Appointment a
                    JOIN Patient p ON a.patient_id = p.patient_id
                    JOIN Doctor d ON a.doctor_id = d.doctor_id
                    WHERE a.doctor_id = ? AND a.appointment_date = ?
                """, (doctor_id, appointment_date.strftime('%Y-%m-%d')))

            if appointments:
                df = pd.DataFrame(appointments, columns=['Appointment ID', 'Patient Name', 'Doctor Name', 'Appointment Date', 'Status'])
                st.dataframe(df)
            elif selected_doctor_name != "Select a Doctor":
                st.write("No appointments found for the selected doctor and date.")
            else:
                st.write("Please select a doctor to view appointments.")
            conn.close()
        else:
            st.error("Failed to connect to the database.")

    elif page == "Admissions":
        st.header("Admissions")
        conn = db_utils.create_connection()
        if conn:
            room_types = [row[0] for row in db_utils.fetch_data(conn, "SELECT DISTINCT type FROM Room")]
            selected_room_type = st.selectbox("Select Room Type", ["All"] + room_types)

            if selected_room_type != "All":
                admissions = db_utils.fetch_data(conn, """
                    SELECT
                        Admission.admission_id,
                        Patient.name,
                        Room.room_number,
                        Admission.admission_date,
                        Admission.discharge_date,
                        Room.type,
                        Room.status
                    FROM Admission
                    JOIN Room ON Admission.room_id = Room.room_id
                    JOIN Patient ON Admission.patient_id = Patient.patient_id
                    WHERE Room.type = ?
                """, (selected_room_type,))
            else:
                admissions = db_utils.fetch_data(conn, """
                    SELECT
                        Admission.admission_id,
                        Patient.name,
                        Room.room_number,
                        Admission.admission_date,
                        Admission.discharge_date,
                        Room.type,
                        Room.status
                    FROM Admission
                    JOIN Room ON Admission.room_id = Room.room_id
                    JOIN Patient ON Admission.patient_id = Patient.patient_id
                """)

            if admissions:
                df = pd.DataFrame(admissions, columns=['Admission ID', 'Patient Name', 'Room Number', 'Admission Date', 'Discharge Date', 'Room Type', 'Room Status'])
                st.dataframe(df)
            else:
                st.write("No admissions found.")
            conn.close()
        else:
            st.error("Failed to connect to the database.")

    # Manage Records GUI must be added here
    # elif page == "Manage Records":
    #     manage_records.app()

    elif page == "Analytics":
        st.header("Analytics")
        conn = db_utils.create_connection()
        if conn:
            occupancy_rate = query_functions.calculate_room_occupancy_rate(conn)
            st.metric("Overall Room Occupancy Rate", f"{occupancy_rate:.2f}%" if occupancy_rate is not None else "N/A")

            room_types = [row[0] for row in db_utils.fetch_data(conn, "SELECT DISTINCT type FROM Room")]
            for r_type in room_types:
                avg_stay = query_functions.compute_average_patient_stay_by_room_type(conn, room_type=r_type)
                st.metric(f"Average Patient Stay ({r_type} Rooms)", f"{avg_stay:.2f} days" if avg_stay is not None else "N/A")

            overbooked_doctors = query_functions.detect_overbooked_doctors(conn)
            st.subheader("Overbooked Doctors:")
            if overbooked_doctors:
                doctor_names = db_utils.fetch_data(conn, "SELECT doctor_id, name FROM Doctor")
                doctor_name_map = {id: name for id, name in doctor_names}
                df = pd.DataFrame(overbooked_doctors, columns=['doctor_id', 'appointment_date', 'appointment_count'])
                df['Doctor Name'] = df['doctor_id'].map(doctor_name_map)
                df = df[['Doctor Name', 'appointment_date', 'appointment_count']]
                st.dataframe(df)
            else:
                st.write("No overbooked doctors detected (threshold: 5 appointments per day per doctor).")

            st.subheader("Visualizations")
            appointments_chart_path = analytics.get_appointments_by_specialization(conn)
            gender_chart_path = analytics.get_patient_gender_distribution(conn)
            room_usage_chart_path = analytics.get_room_usage_trends(conn)

            if appointments_chart_path and os.path.exists(appointments_chart_path):
                st.image(appointments_chart_path, caption="Appointments by Specialization", use_column_width=True)
            else:
                st.write("Appointments by Specialization chart not available.")

            if gender_chart_path and os.path.exists(gender_chart_path):
                st.image(gender_chart_path, caption="Patient Gender Distribution", use_column_width=True)
            else:
                st.write("Patient Gender Distribution chart not available.")

            if room_usage_chart_path and os.path.exists(room_usage_chart_path):
                st.image(room_usage_chart_path, caption="Room Usage Trends", use_column_width=True)
            else:
                st.write("Room Usage Trends chart not available.")
            conn.close()
        else:
            st.error("Failed to connect to the database.")

    elif page == "Export Report":
        st.header("Export Report")
        st.write("Click the button below to generate and download the hospital analytics report as a PDF.")
        conn = db_utils.create_connection()
        if conn:
            analytics.get_appointments_by_specialization(conn)
            analytics.get_patient_gender_distribution(conn)
            analytics.get_room_usage_trends(conn)
            conn.close()

        if st.button("Generate Report"):
            report_path = report_generator.generate_hospital_report()
            if report_path and os.path.exists(report_path):
                with open(report_path, "rb") as file:
                    st.download_button(
                        label="Download Hospital Report PDF",
                        data=file,
                        file_name="hospital_report.pdf",
                        mime="application/pdf",
                    )
            else:
                st.error("Failed to generate the report. Please ensure charts were successfully generated in the Analytics section.")

if __name__ == "__main__":
    main()


Writing gui/dashboard.py


### 📊 **Overview of `gui/dashboard.py`**

This script defines a **Streamlit web application** serving as a complete **hospital analytics and management dashboard**.  
It connects to a **SQLite** database to retrieve and visualize data, manage appointments and admissions, perform analytics, and generate PDF reports.

---

### 🧱 **Structure and Navigation**

The app provides five main sections via the sidebar:

| Page              | Description                                                                       |
| ----------------- | --------------------------------------------------------------------------------- |
| **Dashboard**     | Displays KPIs like number of patients, doctors, and rooms.                        |
| **Appointments**  | Filter and view doctor appointments by date and doctor name.                      |
| **Admissions**    | Shows admission records, filterable by room type.                                 |
| **Analytics**     | Shows calculated KPIs (room occupancy, average stay), overbooked doctors, charts. |
| **Export Report** | Generates a downloadable PDF analytics report.                                    |

---

### 🔍 **Key Functional Features**

#### 1. **Database Interaction**
- Connects via `create_connection()` and `fetch_data()` from `db_utils.py`
- All SQL queries use **parameterized execution** to prevent SQL injection

#### 2. **Dynamic Metrics**
- Uses `st.metric()` to display real-time KPIs (e.g., patient count, room occupancy)

#### 3. **Data Tables**
- Renders data using `pandas` DataFrames and `st.dataframe()` for interactive viewing

#### 4. **Chart Visualization**
- Calls functions from `analytics.py` and displays charts via `st.image()`

#### 5. **PDF Report Generation**
- Generates a summary PDF using `report_generator.py`
- Provides **file download** through `st.download_button()`

---

### 📦 **Libraries Used**

- `streamlit`: Web app framework  
- `pandas`: Data manipulation  
- `os`, `sys`: Path handling  
- `sqlite3`: Database operations  
- Modules from `app/`: Reusable logic for DB, analytics, and reporting

---

### ✅ **Summary**

| Feature       | Description                                            |
| ------------- | ------------------------------------------------------ |
| Framework     | Streamlit                                              |
| Database      | SQLite                                                 |
| Functionality | Data browsing, analytics, visualization, and reporting |
| Security      | Safe SQL querying via parameterized statements         |
| Output        | On-screen metrics, visual charts, downloadable report  |

> 🧠 **Use Case**: Ideal for hospitals or clinics needing a lightweight, interactive system for real-time data monitoring and automated reporting.


In [10]:
# Create __init__.py in app directory
with open('app/__init__.py', 'w') as f:
  f.write("")

# Create static directories (redundant with initial creation but good practice)
if not os.path.exists('static'):
  os.makedirs('static')
if not os.path.exists('static/charts'):
  os.makedirs('static/charts')
if not os.path.exists('static/reports'):
  os.makedirs('static/reports')

create_db()
create_app_files()

print("Database created and seeded. Application files generated in 'app' and 'gui' directories.")
print("You can now run the Streamlit app using the command in the next cell.")

Tables created.
Seeding data...
Data seeding complete.
Database created and seeded. Application files generated in 'app' and 'gui' directories.
You can now run the Streamlit app using the command in the next cell.


In [11]:
!pip install pyngrok

In [12]:
# Run Streamlit app
import subprocess
    # The Streamlit app will open in a new tab or pane, depending on your environment
    # In Colab, you might need to expose the port or use ngrok
    # For a simple run within Colab, you can use this:
subprocess.Popen(['streamlit', 'run', 'gui/dashboard.py'])

<Popen: returncode: None args: ['streamlit', 'run', 'gui/dashboard.py']>

In [ ]:
# Run Streamlit app and expose it via ngrok
import subprocess
from pyngrok import ngrok
import time
import os

# Kill any running ngrok tunnels
ngrok.kill()

# Set your ngrok authentication token
ngrok.set_auth_token(os.environ["NGROK_AUTH_TOKEN"])  # set in Colab secrets / env, never hardcode

# Start the Streamlit app in the background
process = subprocess.Popen(['streamlit', 'run', 'gui/dashboard.py', '--server.port', '8501'])

# Wait a few seconds for Streamlit to start
time.sleep(5)

# Start the ngrok tunnel
try:
    public_url = ngrok.connect(addr=8501, proto="http")
    print(f"✅ Streamlit app running at: {public_url}")

    while True:
        time.sleep(1)
except Exception as e:
    print(f"❌ Error starting ngrok tunnel: {e}")
    process.terminate()


### 🧭 **Overview of the Code**

This Python script launches a **Streamlit dashboard** (served on `localhost:8501`) and exposes it publicly using **Ngrok**, which creates a secure tunnel to the local server.

---

### ⚙️ **Key Functional Steps**

| Step                        | Description                                                               |
| --------------------------- | ------------------------------------------------------------------------- |
| `ngrok.kill()`              | Terminates any existing Ngrok tunnels to prevent conflicts.               |
| `ngrok.set_auth_token(...)` | Authenticates the session with Ngrok using a personal token.              |
| `subprocess.Popen(...)`     | Starts the Streamlit app (`dashboard.py`) in the background on port 8501. |
| `time.sleep(5)`             | Waits 5 seconds to allow the Streamlit server to initialize.              |
| `ngrok.connect(...)`        | Opens an HTTP tunnel to expose the local app to the internet.             |
| `print(...)`                | Displays the public Ngrok URL in the console.                             |
| `while True: time.sleep(1)` | Keeps the script running to maintain the tunnel.                          |
| `except` block              | Handles errors and terminates the Streamlit process if Ngrok fails.       |

---

### ✅ **Usage Benefits**

- 🚫 No need for public IP or port forwarding  
- 🌐 Accessible from any device using the Ngrok URL  
- 💼 Ideal for **demos, testing, and remote collaboration**

---

### 🔐 **Security Note**

Your **Ngrok auth token** should be handled securely:

- Store it in a `.env` file or access via:  
  `os.environ['NGROK_AUTH_TOKEN']`
- ❗ Never hardcode tokens in public scripts

---

### 📌 **Example Output**

```bash
✅ Streamlit app running at: http://<random-id>.ngrok.io




---



---



---



---



### ✅ Recommendations to Improve the Dataset Layer for a More Professional Application

1. **Use an ORM (Object-Relational Mapper)**  
   Replace direct SQL queries with an ORM like **SQLAlchemy**.  
   - Enhances **readability**, **maintainability**, and **security**  
   - Reduces risk of **SQL injection**  
   - Allows for more intuitive data manipulation using Python objects

2. **Implement Data Validation**  
   Validate data before inserting into the database using libraries like:  
   - [`Pydantic`](https://docs.pydantic.dev/)  
   - [`Cerberus`](https://docs.python-cerberus.org/en/stable/)  
   Ensures data **consistency** and **integrity**

3. **Use Database Migrations**  
   Manage database schema changes with tools like **Alembic**  
   - Allows for **controlled schema evolution**  
   - Simplifies **version control** of the database

4. **Separate Data Access Logic**  
   Create a dedicated module or class for all database interactions  
   - Improves **modularity**  
   - Facilitates **unit testing** and **code reuse**

5. **Add Unit Tests**  
   Write unit tests for the data access layer using tools like `pytest`  
   - Ensures code **reliability**  
   - Helps catch issues early during development

6. **Use Environment Variables**  
   Store database credentials and other sensitive info in environment variables  
   - Enhances **security**  
   - Avoids exposing secrets in source code
